In [23]:
import pandas as pd

df = pd.read_csv('Global_Superstore2.csv', encoding='latin1')
print("Shape:", df.shape)
df.head()
df.columns

Shape: (51290, 24)


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country',
       'Postal Code', 'Market', 'Region', 'Product ID', 'Category',
       'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount',
       'Profit', 'Shipping Cost', 'Order Priority'],
      dtype='object')

In [24]:
# 1. Fix data types
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')

In [25]:
# 2. Remove exact duplicate rows

df = df.drop_duplicates()


In [26]:
# 3. Standardize text columns
text_cols = ['Customer Name','City','State','Country','Market','Region',
             'Category','Sub-Category','Ship Mode','Segment','Order Priority']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()


In [27]:
#  4. Handle missing values
df['Postal Code Available'] = df['Postal Code'].notnull()


In [28]:
# 5. Create calculated fields
df['Delivery Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month_name()

In [29]:
# 6. Categorize profit margin into business-friendly buckets
def margin_bucket(x):
    if x < 0: return 'Loss'
    elif x < 10: return 'Low Margin'
    elif x < 25: return 'Medium Margin'
    else: return 'High Margin'
df['Margin Category'] = df['Profit Margin %'].apply(margin_bucket)

In [30]:
# 7. Identify outliers in Profit using the IQR method
Q1 = df['Profit'].quantile(0.25)
Q3 = df['Profit'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
df['Profit Outlier Flag'] = ((df['Profit'] < lower) | (df['Profit'] > upper))

print("Final shape:", df.shape)
print(df['Margin Category'].value_counts())
print("Outliers flagged:", df['Profit Outlier Flag'].sum())

Final shape: (51290, 31)
Margin Category
High Margin      19979
Loss             12515
Medium Margin    10780
Low Margin        8016
Name: count, dtype: int64
Outliers flagged: 9755


In [31]:
df.to_csv('Processed_Global_Superstore.csv', index=False)
files.download('Processed_Global_Superstore.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>